# CerviGrade: retrain the three CNNs on CRIC
Retrains EfficientNetB0, ConvNeXt-Tiny and RegNetY-400MF on CRIC, three seeds each, from the SIPaKMeD checkpoints. Early stopping uses validation macro F1 and the loss is class-weighted. The test set is never used to choose anything.

**Inputs to add:** the CRIC dataset (`cervigrade-cric-dataset`) and the Stage 1 `Training notebook` output. GPU on. No Internet needed.

**To run it unattended:** set `QUICK_TEST = False`, then Save Version, then Save & Run All. Each model has its own cell below, so you can read one model's result at a time.

## 1. Settings
The only cell you edit. Set `QUICK_TEST = True` first to check everything runs (a few minutes), then set it back to `False`.

In [ ]:
# ----- Retrain settings (the only cell you edit) -----
SEEDS = [1, 2, 3]            # one full training run per seed, for each model
RT_EPOCHS = 30               # upper limit; early stopping usually ends sooner
RT_PATIENCE = 5              # stop after this many epochs without a better validation macro F1
RT_LR = 1e-4
RT_DROPOUT = 0.3
CLASS_WEIGHT_POWER = 0.5     # 0 = off, 1 = full balancing. In between is a compromise, because full balancing can hurt precision.
RT_IMG = 224
RT_BATCH = 32
RT_WORKERS = 2
QUICK_TEST = False           # True = one seed and one epoch per model, to check that everything runs

if QUICK_TEST:
    SEEDS, RT_EPOCHS, RT_PATIENCE = [1], 1, 1
    print("QUICK TEST: one seed and one epoch per model. The scores mean nothing.")
RT_ID = f"w{CLASS_WEIGHT_POWER}_e{RT_EPOCHS}_p{RT_PATIENCE}"
RT_DIR = f"/kaggle/working/runs_{RT_ID}"
print("Finished runs are saved in", RT_DIR)

## 2. Find the data and build the split
Same split as the main notebook: by source slide, seed 42, 70 / 15 / 15. The test count should be 1,723.

In [ ]:
import os, re, random
import pandas as pd

def find_containing(root, predicate, max_matches=5):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        if predicate(dirpath, dirnames, filenames):
            matches.append(dirpath)
            if len(matches) >= max_matches:
                break
    return matches

INPUT_ROOT = "/kaggle/input"
cric_matches = find_containing(INPUT_ROOT, lambda d, dn, fn: os.path.basename(d) == "CRIC_cropped")
print("CRIC_cropped found at:", cric_matches)
CRIC_CROPPED_DIR = cric_matches[0]

ckpt_files = {"efficientnetb0": "efficientnetb0_sipakmed_best.keras",
              "convnext": "convnext_sipakmed_best.keras",
              "regnet": "regnet_sipakmed_best.pt"}
checkpoints = {}
for key, fname in ckpt_files.items():
    hits = find_containing(INPUT_ROOT, lambda d, dn, fn, f=fname: f in fn, max_matches=1)
    checkpoints[key] = os.path.join(hits[0], fname) if hits else None
    print(f"  {key}: {'found' if hits else 'MISSING'} -> {checkpoints[key]}")
assert all(checkpoints.values()), "Add the Stage 1 'training-notebook' output as an input."

RANDOM_SEED = 42
def build_cric_manifest():
    rows = []
    for cls in sorted(os.listdir(CRIC_CROPPED_DIR)):
        cls_dir = os.path.join(CRIC_CROPPED_DIR, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            slide_id = fname.split("_")[0]
            rows.append({"filepath": os.path.join(cls_dir, fname), "label": cls,
                         "dataset": "CRIC", "group": f"CRIC_slide_{slide_id}"})
    return pd.DataFrame(rows)

def split_by_group_global(manifest_df, seed=RANDOM_SEED, train_frac=0.7, val_frac=0.15):
    rng = random.Random(seed)
    groups = sorted(manifest_df["group"].unique())
    rng.shuffle(groups)
    n = len(groups)
    n_train = round(n * train_frac)
    n_val = round(n * val_frac)
    group_split = {}
    for g in groups[:n_train]:
        group_split[g] = "train"
    for g in groups[n_train:n_train + n_val]:
        group_split[g] = "val"
    for g in groups[n_train + n_val:]:
        group_split[g] = "test"
    return manifest_df["group"].map(group_split)

cric_manifest = build_cric_manifest()
cric_manifest["split"] = split_by_group_global(cric_manifest)

## 3. Class weights, scoring and helpers
Rare grades get a larger weight in the loss, computed from the training split only. This cell also holds the small helpers the training cells use.

In [ ]:
import os, time, json, copy, gc, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, f1_score

missing = [n for n in ("CRIC_CROPPED_DIR", "checkpoints", "cric_manifest") if n not in globals()]
assert not missing, f"Run the cells above first. Missing: {missing}"
os.makedirs(RT_DIR, exist_ok=True)

CKPT = {"EfficientNetB0": checkpoints["efficientnetb0"], "ConvNeXt": checkpoints["convnext"], "RegNet": checkpoints["regnet"]}
for arch, path in CKPT.items():
    assert os.path.isfile(path), f"Missing Stage 1 checkpoint for {arch}: {path}"

cric_classes = sorted(cric_manifest["label"].unique())
cric_class_to_idx = {c: i for i, c in enumerate(cric_classes)}
SHORT_NAMES = [c.replace("Negative for intraepithelial lesion", "Negative") for c in cric_classes]
RT_K = len(cric_classes)
cric_train_df = cric_manifest[cric_manifest["split"] == "train"]
cric_val_df = cric_manifest[cric_manifest["split"] == "val"]
cric_test_df = cric_manifest[cric_manifest["split"] == "test"]
rt_y_train = cric_train_df["label"].map(cric_class_to_idx).values
rt_y_val = cric_val_df["label"].map(cric_class_to_idx).values
y_true = cric_test_df["label"].map(cric_class_to_idx).values
rt_test_keys = (cric_test_df["label"] + "/" + cric_test_df["filepath"].map(os.path.basename)).values
print(f"CRIC  train {len(cric_train_df)}  val {len(cric_val_df)}  test {len(cric_test_df)}   (test should be 1,723)")

# Rare grades get a larger weight in the loss. The weights come from the training split only.
rt_counts = np.bincount(rt_y_train, minlength=RT_K)
rt_w = (len(rt_y_train) / (RT_K * rt_counts)) ** CLASS_WEIGHT_POWER
rt_w = rt_w * len(rt_y_train) / (rt_w * rt_counts).sum()          # keep the average weight per image at 1
RT_CLASS_WEIGHTS = {i: float(rt_w[i]) for i in range(RT_K)}
print(pd.DataFrame({"train cells": rt_counts, "weight": np.round(rt_w, 2)}, index=SHORT_NAMES))

def rt_test_metrics(y_true, y_prob):
    y_pred = np.argmax(y_prob, axis=1)
    rep = classification_report(y_true, y_pred, target_names=SHORT_NAMES, output_dict=True, zero_division=0)
    out = {"accuracy": round(accuracy_score(y_true, y_pred), 3),
           "weighted_f1": round(rep["weighted avg"]["f1-score"], 3),
           "macro_f1": round(rep["macro avg"]["f1-score"], 3)}
    for c in SHORT_NAMES:
        out[f"recall_{c}"] = round(rep[c]["recall"], 3)
    return out

RT_RUNS_FILE = os.path.join(RT_DIR, "runs.json")
rt_runs = json.load(open(RT_RUNS_FILE)) if os.path.exists(RT_RUNS_FILE) else {}
def rt_probs_path(arch, seed): return os.path.join(RT_DIR, f"probs_{arch}_seed{seed}.npz")
def rt_weights_path(arch, seed): return os.path.join(RT_DIR, f"weights_{arch}_seed{seed}" + (".pt" if arch == "RegNet" else ".weights.h5"))
def rt_run_done(arch, seed): return f"{arch}_seed{seed}" in rt_runs and os.path.exists(rt_probs_path(arch, seed))
print(f"{len(rt_runs)} finished run(s) found on disk.")

# ----- Helpers used by the three training cells -----
from IPython.display import display
RT_METRIC_COLS = ["accuracy", "weighted_f1", "macro_f1"] + [f"recall_{c}" for c in SHORT_NAMES]

def rt_mean_std(series):
    m, s = series.mean(), series.std(ddof=1)
    return f"{m:.3f} ± {s:.3f}" if len(series) > 1 else f"{m:.3f} (1 run)"

def rt_train_all(arch):
    """Train one model once per seed. Finished runs are skipped, so it is safe to run again."""
    durations = []
    for n, seed in enumerate(SEEDS, 1):
        key = f"{arch}_seed{seed}"
        if rt_run_done(arch, seed):
            print(f"[{n}/{len(SEEDS)}] {key}: already finished, skipping")
            continue
        t0 = time.time()
        info, probs = rt_run_regnet(seed) if arch == "RegNet" else rt_run_keras(arch, seed)
        minutes = (time.time() - t0) / 60
        durations.append(minutes)
        np.savez(rt_probs_path(arch, seed), y_prob=probs, y_true=y_true, test_keys=rt_test_keys)
        rt_runs[key] = {"architecture": arch, "seed": seed, "minutes": round(minutes, 1), **info, **rt_test_metrics(y_true, probs)}
        json.dump(rt_runs, open(RT_RUNS_FILE, "w"), indent=2)
        r = rt_runs[key]
        left = sum(1 for s in SEEDS[n:] if not rt_run_done(arch, s))
        print(f"[{n}/{len(SEEDS)}] {key}: {minutes:.1f} min | best epoch {r['best_epoch']} of {r['epochs_run']} | "
              f"val macro F1 {r['val_macro_f1']} | test acc {r['accuracy']} weighted F1 {r['weighted_f1']} "
              f"macro F1 {r['macro_f1']} SCC recall {r['recall_SCC']} | about {left * np.mean(durations):.0f} min left")
    rt_show(arch)

def rt_show(arch):
    sub = pd.DataFrame([r for r in rt_runs.values() if r["architecture"] == arch]).sort_values("seed")
    if QUICK_TEST:
        print("QUICK TEST: these scores come from one epoch and mean nothing.")
    print(f"\n{arch}: every seed")
    display(sub[["seed", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS].reset_index(drop=True))
    print(f"{arch}: average ± standard deviation across seeds")
    display(pd.DataFrame({c: [rt_mean_std(sub[c])] for c in RT_METRIC_COLS}, index=[f"{arch} ({len(sub)} runs)"]))

## 4. Keras training code (EfficientNetB0 and ConvNeXt)
Definitions only. Nothing trains here.

In [ ]:
import tensorflow as tf
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.set_logical_device_configuration(gpu, [tf.config.LogicalDeviceConfiguration(memory_limit=10240)])
except Exception as e:
    print("GPU memory limit left as it was:", e)    # harmless; TensorFlow was already started by an earlier cell
from tensorflow.keras import layers, models
from tensorflow.keras.applications.efficientnet import preprocess_input as rt_effnet_pre
from tensorflow.keras.applications.convnext import preprocess_input as rt_convnext_pre
RT_PREPROCESS = {"EfficientNetB0": rt_effnet_pre, "ConvNeXt": rt_convnext_pre}

def rt_make_tf_dataset(df, preprocess_fn, shuffle_seed=None, augment=False):
    filepaths = df["filepath"].values
    labels = df["label"].map(cric_class_to_idx).values
    def load_and_preprocess(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [RT_IMG, RT_IMG])
        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
        return preprocess_fn(image), label
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle_seed is not None:
        ds = ds.shuffle(buffer_size=len(df), seed=shuffle_seed)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(RT_BATCH).prefetch(tf.data.AUTOTUNE)

def rt_build_keras(arch):
    loaded = tf.keras.models.load_model(CKPT[arch])
    base = next(l for l in loaded.layers if isinstance(l, tf.keras.Model))
    base.trainable = True
    inputs = layers.Input(shape=(RT_IMG, RT_IMG, 3))
    x = base(inputs)
    x = layers.Dropout(RT_DROPOUT)(x)
    outputs = layers.Dense(RT_K, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=RT_LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

class MacroF1Stopping(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val, patience):
        super().__init__()
        self.val_ds, self.y_val, self.patience = val_ds, y_val, patience
        self.best, self.best_epoch, self.best_weights, self.wait = -1.0, 0, None, 0
    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)
        f1 = f1_score(self.y_val, probs.argmax(1), average="macro", zero_division=0)
        logs["val_macro_f1"] = f1
        if f1 > self.best + 1e-6:
            self.best, self.best_epoch, self.wait = f1, epoch + 1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

class RtEpochBar(tf.keras.callbacks.Callback):
    def __init__(self, max_epochs, title):
        super().__init__()
        self.max_epochs, self.title = max_epochs, title
    def on_train_begin(self, logs=None):
        self.outer = tqdm(total=self.max_epochs, desc=self.title, unit="epoch")
    def on_epoch_begin(self, epoch, logs=None):
        self.inner = tqdm(total=self.params.get("steps"), desc=f"  epoch {epoch + 1}", unit="batch", leave=False)
    def on_train_batch_end(self, batch, logs=None):
        self.inner.update(1)
    def on_epoch_end(self, epoch, logs=None):
        self.inner.close()
        self.outer.update(1)
        self.outer.set_postfix(loss=f"{logs['loss']:.3f}", acc=f"{logs['accuracy']:.3f}", val_macro_f1=f"{logs['val_macro_f1']:.3f}")
    def on_train_end(self, logs=None):
        self.outer.close()

def rt_run_keras(arch, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    pre = RT_PREPROCESS[arch]
    train_ds = rt_make_tf_dataset(cric_train_df, pre, shuffle_seed=seed, augment=True)
    val_ds = rt_make_tf_dataset(cric_val_df, pre)
    test_ds = rt_make_tf_dataset(cric_test_df, pre)
    model = rt_build_keras(arch)
    stopper = MacroF1Stopping(val_ds, rt_y_val, RT_PATIENCE)
    history = model.fit(train_ds, epochs=RT_EPOCHS, class_weight=RT_CLASS_WEIGHTS, verbose=0,
                        callbacks=[stopper, RtEpochBar(RT_EPOCHS, f"{arch} seed {seed}")])
    probs = model.predict(test_ds, verbose=0)
    model.save_weights(rt_weights_path(arch, seed))
    info = {"best_epoch": stopper.best_epoch, "epochs_run": len(history.history["loss"]), "val_macro_f1": round(stopper.best, 3)}
    del model, train_ds, val_ds, test_ds
    tf.keras.backend.clear_session(); gc.collect()
    return info, probs

## 5. Train EfficientNetB0
One run per seed, then its table. Finished runs are saved, so running the cell again skips them.

In [ ]:
rt_train_all("EfficientNetB0")

## 6. Train ConvNeXt-Tiny

In [ ]:
rt_train_all("ConvNeXt")

## 7. RegNetY-400MF training code (PyTorch)
Definitions only. Nothing trains here.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import regnet_y_400mf
from PIL import Image

rt_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", rt_device)

_mean, _std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
rt_train_tf = transforms.Compose([transforms.Resize((RT_IMG, RT_IMG)), transforms.RandomHorizontalFlip(),
                                  transforms.RandomVerticalFlip(), transforms.ToTensor(), transforms.Normalize(_mean, _std)])
rt_eval_tf = transforms.Compose([transforms.Resize((RT_IMG, RT_IMG)), transforms.ToTensor(), transforms.Normalize(_mean, _std)])

class RtCricDataset(Dataset):
    def __init__(self, df, transform):
        self.filepaths = df["filepath"].values
        self.labels = df["label"].map(cric_class_to_idx).values
        self.transform = transform
    def __len__(self):
        return len(self.filepaths)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.filepaths[idx]).convert("RGB")), int(self.labels[idx])

def rt_build_regnet():
    model = regnet_y_400mf(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, 5)          # the 5-class SIPaKMeD head, so the saved weights fit
    model.load_state_dict(torch.load(CKPT["RegNet"], map_location="cpu"))
    model.fc = torch.nn.Sequential(torch.nn.Dropout(RT_DROPOUT), torch.nn.Linear(model.fc.in_features, RT_K))
    for p in model.parameters():
        p.requires_grad = True
    return model.to(rt_device)

def rt_torch_probs(model, loader, desc):
    model.eval()
    out = []
    with torch.no_grad():
        for images, _ in tqdm(loader, desc=desc, unit="batch", leave=False):
            out.append(torch.softmax(model(images.to(rt_device)), dim=1).cpu().numpy())
    return np.concatenate(out)

def rt_run_regnet(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(RtCricDataset(cric_train_df, rt_train_tf), batch_size=RT_BATCH, shuffle=True, num_workers=RT_WORKERS, generator=g)
    val_loader = DataLoader(RtCricDataset(cric_val_df, rt_eval_tf), batch_size=RT_BATCH, shuffle=False, num_workers=RT_WORKERS)
    test_loader = DataLoader(RtCricDataset(cric_test_df, rt_eval_tf), batch_size=RT_BATCH, shuffle=False, num_workers=RT_WORKERS)
    model = rt_build_regnet()
    optimizer = torch.optim.Adam(model.parameters(), lr=RT_LR)
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([RT_CLASS_WEIGHTS[i] for i in range(RT_K)], dtype=torch.float32, device=rt_device))
    best, best_epoch, best_state, wait, epochs_run = -1.0, 0, None, 0, 0
    bar = tqdm(range(RT_EPOCHS), desc=f"RegNet seed {seed}", unit="epoch")
    for epoch in bar:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"  epoch {epoch + 1}", unit="batch", leave=False):
            images, labels = images.to(rt_device), labels.to(rt_device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
        epochs_run += 1
        f1 = f1_score(rt_y_val, rt_torch_probs(model, val_loader, "  validating").argmax(1), average="macro", zero_division=0)
        bar.set_postfix(loss=f"{total_loss / total:.3f}", acc=f"{correct / total:.3f}", val_macro_f1=f"{f1:.3f}")
        if f1 > best + 1e-6:
            best, best_epoch, wait = f1, epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= RT_PATIENCE:
                break
    model.load_state_dict(best_state)
    probs = rt_torch_probs(model, test_loader, "testing")
    torch.save(model.state_dict(), rt_weights_path("RegNet", seed))
    info = {"best_epoch": best_epoch, "epochs_run": epochs_run, "val_macro_f1": round(best, 3)}
    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return info, probs

## 8. Train RegNetY-400MF

In [ ]:
rt_train_all("RegNet")

## 9. All three side by side
Every run, then the average ± standard deviation per model. Differences smaller than the spread between seeds are noise.

In [ ]:
# ----- All three models side by side -----
rt_order = {a: i for i, a in enumerate(["EfficientNetB0", "ConvNeXt", "RegNet"])}
rt_df = pd.DataFrame(rt_runs.values())
assert len(rt_df), "No finished runs yet. Run the three training cells above first."
rt_df = rt_df.assign(_o=rt_df["architecture"].map(rt_order)).sort_values(["_o", "seed"]).drop(columns="_o").reset_index(drop=True)
missing_models = [a for a in rt_order if a not in set(rt_df["architecture"])]
if missing_models:
    print("Not trained yet:", missing_models)
if QUICK_TEST:
    print("QUICK TEST: the scores below come from one epoch and mean nothing.")
print("Every run")
display(rt_df[["architecture", "seed", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS])

rt_summary = rt_df.groupby("architecture")[RT_METRIC_COLS].agg(rt_mean_std)
rt_summary = rt_summary.loc[[a for a in rt_order if a in rt_summary.index]]
rt_summary.insert(0, "runs", rt_df.groupby("architecture").size())
print("Average ± standard deviation across seeds")
display(rt_summary)
rt_summary.to_csv("/kaggle/working/retrain_summary.csv")
rt_df.to_csv("/kaggle/working/retrain_all_runs.csv", index=False)

## 10. Pick the seed for each model and save it
The seed with the best **validation** macro F1 is kept, saved, reloaded and checked against its stored test probabilities. The files in `/kaggle/working` are what the main notebook reads.

In [ ]:
# ----- Pick one seed per model by validation score, save it, and check the saved file -----
rt_best_seed = {}
for arch in rt_order:
    sub = rt_df[rt_df["architecture"] == arch]
    if len(sub):
        rt_best_seed[arch] = int(sub.sort_values(["val_macro_f1", "seed"], ascending=[False, True]).iloc[0]["seed"])
print("Chosen seed per model (best validation macro F1):", rt_best_seed)

rt_bundle = {"y_true": y_true, "test_keys": rt_test_keys}
for arch, seed in rt_best_seed.items():
    rt_bundle[arch] = np.load(rt_probs_path(arch, seed))["y_prob"]
    if arch == "RegNet":
        m = rt_build_regnet()
        m.load_state_dict(torch.load(rt_weights_path(arch, seed), map_location=rt_device))
        check = rt_torch_probs(m, DataLoader(RtCricDataset(cric_test_df, rt_eval_tf), batch_size=RT_BATCH, num_workers=RT_WORKERS), "re-checking")
        torch.save(m.state_dict(), "/kaggle/working/regnet_cric_retrained.pt")
        del m
    else:
        tf.keras.backend.clear_session()
        m = rt_build_keras(arch)
        m.load_weights(rt_weights_path(arch, seed))
        check = m.predict(rt_make_tf_dataset(cric_test_df, RT_PREPROCESS[arch]), verbose=0)
        m.save(f"/kaggle/working/{arch.lower()}_cric_retrained.keras")
        del m
    gap = np.abs(check - rt_bundle[arch]).max()
    print(f"{arch}: saved the seed {seed} model. Reloaded model matches the stored test probabilities (largest gap {gap:.4f}).")
    if gap > 0.02:
        print("  WARNING: the reloaded model does not match. Do not use this saved file.")
np.savez("/kaggle/working/stage2_retrain_probs.npz", **rt_bundle)
json.dump({"best_seed": rt_best_seed, "settings_id": RT_ID, "class_weight_power": CLASS_WEIGHT_POWER},
          open("/kaggle/working/retrain_selection.json", "w"), indent=2)